In [1]:
# phase2_multi_city.py
# Goal: Fetch weather data for 15 cities and store it cleanly

import requests
import json
import time    # we'll use this to avoid hammering the API too fast

# --- CONFIGURATION ---
API_KEY = "e365ed8fdcf3fbe95aba8de8c70d2bc5"
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
UNITS = "metric"

# --- CITIES LIST ---
# Mix of Indian cities + global cities = better analysis + better portfolio
CITIES = [
    # Indian cities
    "Lucknow", "Delhi", "Mumbai", "Bangalore", "Kolkata",
    "Chennai", "Hyderabad", "Jaipur", "Pune", "Ahmedabad",
    # Global cities
    "London", "New York", "Tokyo", "Dubai", "Sydney"
]

In [2]:
# --- FUNCTION TO FETCH ONE CITY ---
def fetch_weather(city):
    """
    Fetches weather data for a single city.
    Returns a clean dictionary, or None if something goes wrong.
    """
    params = {
        "q": city,
        "appid": API_KEY,
        "units": UNITS
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        # timeout=10 means "give up if no response in 10 seconds"

        if response.status_code == 200:
            data = response.json()

            # Extract only the fields we care about
            weather_info = {
                "city"        : data["name"],
                "country"     : data["sys"]["country"],
                "temperature" : data["main"]["temp"],
                "feels_like"  : data["main"]["feels_like"],
                "temp_min"    : data["main"]["temp_min"],
                "temp_max"    : data["main"]["temp_max"],
                "humidity"    : data["main"]["humidity"],
                "pressure"    : data["main"]["pressure"],
                "wind_speed"  : data["wind"]["speed"],
                "condition"   : data["weather"][0]["main"],
                "description" : data["weather"][0]["description"],
                "visibility"  : data.get("visibility", None),
                # data.get() is safer — returns None if key doesn't exist
            }
            return weather_info

        elif response.status_code == 404:
            print(f"  [SKIPPED] '{city}' not found — check spelling")
            return None

        elif response.status_code == 429:
            print(f"  [WAIT] Rate limit hit — waiting 60 seconds...")
            time.sleep(60)
            return None

        else:
            print(f"  [ERROR] {city} → status code {response.status_code}")
            return None

    except requests.exceptions.Timeout:
        print(f"  [TIMEOUT] {city} took too long to respond")
        return None

    except requests.exceptions.ConnectionError:
        print(f"  [CONNECTION ERROR] Check your internet connection")
        return None

In [3]:
# --- MAIN LOOP ---
all_weather_data = []   # this list will hold one dict per city

print("Starting data collection...\n")
print(f"{'City':<15} {'Temp':>6} {'Humidity':>9} {'Condition'}")
print("-" * 50)

for city in CITIES:
    result = fetch_weather(city)

    if result is not None:                      # only add if successful
        all_weather_data.append(result)

        # Print a live progress update
        print(f"{result['city']:<15} {result['temperature']:>5.1f}°C  "
              f"{result['humidity']:>6}%   {result['condition']}")

    time.sleep(1)   # wait 1 second between calls — be polite to the API!

print("-" * 50)
print(f"\nDone! Collected data for {len(all_weather_data)} cities.")

Starting data collection...

City              Temp  Humidity Condition
--------------------------------------------------
Lucknow          34.0°C      22%   Haze
Delhi            32.0°C      25%   Haze
Mumbai           34.0°C      52%   Smoke
Bengaluru        32.1°C      44%   Clear
Kolkata          32.0°C      70%   Haze
Chennai          33.2°C      60%   Clouds
Hyderabad        35.2°C      31%   Haze
Jaipur           32.6°C      18%   Haze
Pune             37.3°C      10%   Clear
Ahmedabad        33.0°C      25%   Smoke
London            3.7°C      88%   Clear
New York         19.9°C      65%   Clear
Tokyo            23.6°C      49%   Clouds
Dubai            24.9°C      63%   Clouds
Sydney           23.2°C      50%   Clear
--------------------------------------------------

Done! Collected data for 15 cities.


In [4]:
# phase3_pandas_csv.py
# Goal: Convert raw data → clean DataFrame → save as CSV

import requests
import time
import pandas as pd        # the most important library for data analysts
from datetime import datetime  # to timestamp when data was collected

In [5]:
# --- CREATE DATAFRAME ---
df = pd.DataFrame(all_weather_data)
# Pandas automatically uses the dictionary keys as column names!

print("--- RAW DATAFRAME ---")
print(df)
print(f"\nShape: {df.shape}")        # (rows, columns) e.g. (15, 12)
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

--- RAW DATAFRAME ---
         city country  temperature  feels_like  temp_min  temp_max  humidity  \
0     Lucknow      IN        33.99       32.15     33.99     33.99        22   
1       Delhi      IN        32.05       30.37     32.05     32.05        25   
2      Mumbai      IN        33.99       39.06     30.94     33.99        52   
3   Bengaluru      IN        32.09       33.18     31.12     33.06        44   
4     Kolkata      IN        31.97       38.97     31.97     31.97        70   
5     Chennai      IN        33.25       40.16     32.83     33.46        60   
6   Hyderabad      IN        35.23       35.23     35.23     35.73        31   
7      Jaipur      IN        32.62       30.46     32.62     32.62        18   
8        Pune      IN        37.30       34.42     37.30     37.30        10   
9   Ahmedabad      IN        33.02       31.40     33.02     33.02        25   
10     London      GB         3.70        2.55      2.21      5.16        88   
11   New York     

In [6]:
# --- CHECK FOR MISSING VALUES ---
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())
# shows count of missing values per column

print(f"\nTotal missing values: {df.isnull().sum().sum()}")


--- MISSING VALUES ---
city           0
country        0
temperature    0
feels_like     0
temp_min       0
temp_max       0
humidity       0
pressure       0
wind_speed     0
condition      0
description    0
visibility     0
dtype: int64

Total missing values: 0


In [7]:
# --- HANDLE MISSING VALUES ---

# For visibility — fill missing with the column average
df["visibility"] = df["visibility"].fillna(df["visibility"].mean())

# Round it to clean integer (visibility in metres)
df["visibility"] = df["visibility"].round(0).astype(int)

# Verify no more nulls
print(f"\nMissing values after cleaning: {df.isnull().sum().sum()}")
# Should print 0


Missing values after cleaning: 0


In [8]:
# --- RENAME COLUMNS ---
df = df.rename(columns={
    "temperature" : "temp_c",
    "feels_like"  : "feels_like_c",
    "temp_min"    : "temp_min_c",
    "temp_max"    : "temp_max_c",
    "wind_speed"  : "wind_speed_mps",
    "pressure"    : "pressure_hpa",
    "visibility"  : "visibility_m",
})
# Adding units to column names 

print("\nRenamed columns:")
print(df.columns.tolist())


Renamed columns:
['city', 'country', 'temp_c', 'feels_like_c', 'temp_min_c', 'temp_max_c', 'humidity', 'pressure_hpa', 'wind_speed_mps', 'condition', 'description', 'visibility_m']


In [9]:
# --- ADD DERIVED COLUMNS ---

# 1. Feels-like difference (how much hotter/colder it feels vs actual)
df["feels_like_diff"] = (df["feels_like_c"] - df["temp_c"]).round(1)

# 2. Temperature category
def categorize_temp(temp):
    if temp < 15:
        return "Cold"
    elif temp < 25:
        return "Mild"
    elif temp < 33:
        return "Warm"
    else:
        return "Hot"

df["temp_category"] = df["temp_c"].apply(categorize_temp)

# 3. Humidity level
def categorize_humidity(humidity):
    if humidity < 40:
        return "Dry"
    elif humidity < 70:
        return "Comfortable"
    else:
        return "Humid"

df["humidity_level"] = df["humidity"].apply(categorize_humidity)

# 4. Timestamp — when was this data collected?
df["collected_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("\n--- DATAFRAME WITH NEW COLUMNS ---")
print(df[["city", "temp_c", "temp_category", 
          "humidity", "humidity_level", "feels_like_diff"]].to_string())


--- DATAFRAME WITH NEW COLUMNS ---
         city  temp_c temp_category  humidity humidity_level  feels_like_diff
0     Lucknow   33.99           Hot        22            Dry             -1.8
1       Delhi   32.05          Warm        25            Dry             -1.7
2      Mumbai   33.99           Hot        52    Comfortable              5.1
3   Bengaluru   32.09          Warm        44    Comfortable              1.1
4     Kolkata   31.97          Warm        70          Humid              7.0
5     Chennai   33.25           Hot        60    Comfortable              6.9
6   Hyderabad   35.23           Hot        31            Dry              0.0
7      Jaipur   32.62          Warm        18            Dry             -2.2
8        Pune   37.30           Hot        10            Dry             -2.9
9   Ahmedabad   33.02           Hot        25            Dry             -1.6
10     London    3.70          Cold        88          Humid             -1.2
11   New York   19.85       

In [10]:
# --- REORDER COLUMNS ---
# Put the most important columns first — makes CSV easier to read

column_order = [
    "city", "country", "collected_at",
    "temp_c", "feels_like_c", "feels_like_diff", "temp_category",
    "temp_min_c", "temp_max_c",
    "humidity", "humidity_level",
    "pressure_hpa", "wind_speed_mps", "visibility_m",
    "condition", "description"
]

df = df[column_order]
print("\nFinal column order set!")


Final column order set!


In [11]:
# --- SAVE TO CSV ---
filename = f"weather_data_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
# e.g. weather_data_20260414_1530.csv — timestamped filename!

df.to_csv(filename, index=False)
# index=False means don't save the row numbers as a column

print(f"\nCSV saved as: {filename}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")


CSV saved as: weather_data_20260414_1052.csv
Shape: 15 rows × 16 columns


In [12]:
# --- VERIFY THE CSV ---
df_check = pd.read_csv(filename)

print("\n--- VERIFICATION ---")
print(df_check.head())           # first 5 rows
print(f"\nShape matches: {df_check.shape == df.shape}")  # should be True

# Quick summary statistics
print("\n--- SUMMARY STATS ---")
print(df_check[["temp_c", "humidity", "wind_speed_mps"]].describe().round(2))


--- VERIFICATION ---
        city country         collected_at  temp_c  feels_like_c  \
0    Lucknow      IN  2026-04-14 10:50:58   33.99         32.15   
1      Delhi      IN  2026-04-14 10:50:58   32.05         30.37   
2     Mumbai      IN  2026-04-14 10:50:58   33.99         39.06   
3  Bengaluru      IN  2026-04-14 10:50:58   32.09         33.18   
4    Kolkata      IN  2026-04-14 10:50:58   31.97         38.97   

   feels_like_diff temp_category  temp_min_c  temp_max_c  humidity  \
0             -1.8           Hot       33.99       33.99        22   
1             -1.7          Warm       32.05       32.05        25   
2              5.1           Hot       30.94       33.99        52   
3              1.1          Warm       31.12       33.06        44   
4              7.0          Warm       31.97       31.97        70   

  humidity_level  pressure_hpa  wind_speed_mps  visibility_m condition  \
0            Dry          1008            3.09          5000      Haze   
1     